In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score

In [3]:
pd.set_option("display.max_columns", 30)
df = pd.read_csv("../data/raw/german_credit_openml.csv")

In [4]:
df.shape

(1000, 21)

In [6]:
df.head(5)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes,bad


In [7]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
cat_cols.remove("class")

In [8]:
print("rəqəm:", num_cols)
print("kateqorik:", cat_cols)

rəqəm: ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
kateqorik: ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']


In [9]:
y = (df["class"] == "bad").astype(int)
y.value_counts()

class
0    700
1    300
Name: count, dtype: int64

In [10]:
"default rate:", round(y.mean(), 3)

('default rate:', np.float64(0.3))

In [11]:
miss = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
print(miss)

checking_status           0.0
duration                  0.0
credit_history            0.0
purpose                   0.0
credit_amount             0.0
savings_status            0.0
employment                0.0
installment_commitment    0.0
personal_status           0.0
other_parties             0.0
residence_since           0.0
property_magnitude        0.0
age                       0.0
other_payment_plans       0.0
housing                   0.0
existing_credits          0.0
job                       0.0
num_dependents            0.0
own_telephone             0.0
foreign_worker            0.0
class                     0.0
dtype: float64


In [12]:
print((df[cat_cols] == "NA").sum())

checking_status        0
credit_history         0
purpose                0
savings_status         0
employment             0
personal_status        0
other_parties          0
property_magnitude     0
other_payment_plans    0
housing                0
job                    0
own_telephone          0
foreign_worker         0
dtype: int64


In [13]:
print("dublikat sətir:", df.duplicated().sum())

dublikat sətir: 0


In [14]:
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class


In [15]:
for c in cat_cols:
    print(f"{c}: {df[c].value_counts(dropna=False).to_dict()}")

checking_status: {'no checking': 394, '<0': 274, '0<=X<200': 269, '>=200': 63}
credit_history: {'existing paid': 530, 'critical/other existing credit': 293, 'delayed previously': 88, 'all paid': 49, 'no credits/all paid': 40}
purpose: {'radio/tv': 280, 'new car': 234, 'furniture/equipment': 181, 'used car': 103, 'business': 97, 'education': 50, 'repairs': 22, 'domestic appliance': 12, 'other': 12, 'retraining': 9}
savings_status: {'<100': 603, 'no known savings': 183, '100<=X<500': 103, '500<=X<1000': 63, '>=1000': 48}
employment: {'1<=X<4': 339, '>=7': 253, '4<=X<7': 174, '<1': 172, 'unemployed': 62}
personal_status: {'male single': 548, 'female div/dep/mar': 310, 'male mar/wid': 92, 'male div/sep': 50}
other_parties: {'none': 907, 'guarantor': 52, 'co applicant': 41}
property_magnitude: {'car': 332, 'real estate': 282, 'life insurance': 232, 'no known property': 154}
other_payment_plans: {'none': 814, 'bank': 139, 'stores': 47}
housing: {'own': 713, 'rent': 179, 'for free': 108}
job:

In [16]:
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
duration,1000.0,20.903,12.058814,4.0,12.0,18.0,24.00,72.0
credit_amount,1000.0,3271.258,2822.736876,250.0,1365.5,2319.5,3972.25,18424.0
installment_commitment,1000.0,2.973,1.118715,1.0,2.0,3.0,4.00,4.0
residence_since,1000.0,2.845,1.103718,1.0,2.0,3.0,4.00,4.0
age,1000.0,35.546,11.375469,19.0,27.0,33.0,42.00,75.0
existing_credits,1000.0,1.407,0.577654,1.0,1.0,1.0,2.00,4.0
num_dependents,1000.0,1.155,0.362086,1.0,1.0,1.0,1.00,2.0


In [17]:
checks = {
    "yaş <= 0": (df["age"] <= 0).sum(),
    "müddət <= 0": (df["duration"] <= 0).sum(),
    "kredit məbləği <= 0": (df["credit_amount"] <= 0).sum(),
}
checks

{'yaş <= 0': np.int64(0),
 'müddət <= 0': np.int64(0),
 'kredit məbləği <= 0': np.int64(0)}

In [18]:
rows = []
for c in num_cols:
    auc = roc_auc_score(y, df[c])
    rows.append({
        "feature": c,
        "auc": round(auc, 3),
        "gini_single": round(2 * max(auc, 1 - auc) - 1, 3),
    })

leak_num = pd.DataFrame(rows).sort_values("gini_single", ascending=False)
leak_num

,feature,auc,gini_single
0,duration,0.629,0.257
4,age,0.429,0.141
1,credit_amount,0.555,0.110
2,installment_commitment,0.543,0.087
5,existing_credits,0.475,0.050
3,residence_since,0.502,0.003
6,num_dependents,0.499,0.002


In [19]:
rows = []
for c in cat_cols:
    for lvl in df[c].dropna().unique():
        ind = (df[c] == lvl).astype(int)
        if ind.nunique() < 2:
            continue
        auc = roc_auc_score(y, ind)
        rows.append({
            "feature": f"{c}={lvl}",
            "auc": round(auc, 3),
            "gini_single": round(2 * max(auc, 1 - auc) - 1, 3),
        })


In [20]:
leak_cat = pd.DataFrame(rows).sort_values("gini_single", ascending=False)
leak_cat.head(10)

,feature,auc,gini_single
2,checking_status=no checking,0.328,0.344
0,checking_status=<0,0.626,0.251
4,credit_history=critical/other existing credit,0.410,0.180
20,savings_status=<100,0.586,0.172
43,housing=own,0.434,0.133
36,property_magnitude=real estate,0.441,0.117
1,checking_status=0<=X<200,0.558,0.116
19,savings_status=no known savings,0.445,0.109
9,purpose=radio/tv,0.448,0.105
38,property_magnitude=no known property,0.550,0.099


In [21]:
df.groupby("class")[num_cols].mean().round(1)

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents
class,,,,,,,
bad,24.9,3938.1,3.1,2.8,34.0,1.4,1.2
good,19.2,2985.5,2.9,2.8,36.2,1.4,1.2
